In [ ]:
import os
import time
import json
from pathlib import Path

from tuutrag.qdrant import VectorDB
from qdrant_client import models
from utils import load_config
from dotenv import load_dotenv
from tuutrag.batches import *
from concurrent.futures import ThreadPoolExecutor, as_completed

from google import genai
from google.genai import types
from openai import OpenAI

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
load_dotenv()

openai_key = os.getenv("OPENAI_API_KEY")
gemini_key = os.getenv("GEMINI_API_KEY")

In [92]:
config = load_config("config.ini")

# Connect to QDRANT
qdrant_client = VectorDB(
    port=config.getint("QDRANT", "port"),
    host=config.get("QDRANT", "host")
)

[2026-02-24 22:22:39] Connected to http://localhost:6333/dashboard
[2026-02-24 22:22:39] Storage will be at /Users/pablobedolla/DlrowSreckah/tuutrag-open/tuutrag/data/qdrant


In [93]:
# Connect to OpenAI
openai_api = OpenAI(api_key=openai_key)

# Connect to Gemini
gemini_api = genai.Client(api_key=gemini_key)

In [ ]:
# Create branch chunk collection
qdrant_client.create_collection(
    collection_name="branch_chunks",
    vector_params=models.VectorParams(
        size=3072,
        distance=models.Distance.COSINE
    )
)

In [ ]:
GEMINI_MODEL = "gemini-embedding-001"
TASK_TYPE = "SEMANTIC_SIMILARITY"
BATCH_MAX = 1500

In [ ]:
with open("./mock.json", "r", encoding="utf-8") as f:
    mock_data = json.load(f)
    text_data = [j['chunk'] for j in mock_data]

In [ ]:
BATCH_SIZE = 1500
batches = [mock_data[i:i + BATCH_SIZE] for i in range(0, len(mock_data), BATCH_SIZE)]

print(f"Total records : {len(mock_data)}")
print(f"Total batches : {len(batches)} (max {BATCH_SIZE} each)")

In [ ]:
def process_batch(batch_index: int, batch: list) -> str:
    requests = [
        create_request(
            text=item["chunk"],
            uuid=item["uuid"],
        )
        for item in batch
    ]
    file_name = f"batch_{batch_index:04d}"
    write_jsonl(requests, file_name)
    return file_name

In [ ]:
with ThreadPoolExecutor() as executor:
    futures = {
        executor.submit(process_batch, i, batch): i
        for i, batch in enumerate(batches)
    }

    for future in as_completed(futures):
        batch_idx = futures[future]
        try:
            result = future.result()
            print(f"✓ Batch {batch_idx:04d} → {result}.jsonl")
        except Exception as e:
            print(f"✗ Batch {batch_idx:04d} failed: {e}")

In [ ]:
BATCH_DIR = Path.cwd().parent / "data/gemini_batches/"
batch_jobs = []

In [ ]:
result = gemini_api.models.embed_content(
    model="gemini-embedding-001",
    contents=text_data,
)

In [94]:
from uuid import UUID

points = [
    models.PointStruct(
        id=str(UUID(str(mock_data[idx]["uuid"]).replace(".", "-"))),
        vector=emb.values,
        payload={"text": text_data[idx]},
    )
    for idx, emb in enumerate(result.embeddings)
]

qdrant_client.client.upsert(
    collection_name="branch_chunks",
    points=points,
)

ValueError: badly formed hexadecimal UUID string